In [1]:
from pathlib import Path
import os
import time
import random
import torch

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_MLP import SingleOutMLPNet
from metrics import eval_single_metrics_test

In [2]:
SEED = 12
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


In [3]:
OUTDIR = Path(f"./Results_MLP/seed_{SEED}")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path(f"./trained_models_MLP/seed_{SEED}")
MODELSDIR.mkdir(parents=True, exist_ok=True)

#### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

##### Dataframes

In [7]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s, y_test_s, BATCH_SIZE)

### Training

In [8]:
EPOCHS = 150
METRICS_EVERY = int(EPOCHS/15)
LR = 1e-3
WD = 1e-6

profile_rows = []

In [9]:
IN_DIM = X_train_s.shape[1]
HIDDEN = (512, 256, 128)

In [10]:
torch.manual_seed(SEED)    
model_single = SingleOutMLPNet(IN_DIM, hidden=HIDDEN).to(DEVICE)    
loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)
    
# train_single(model_single, loss, optimizer, train_loader, val_loader, 
#              device=DEVICE, savedir=OUTDIR, 
#              lr=LR, weight_decay=WD, epochs=EPOCHS, 
#              metrics_every=METRICS_EVERY, sex_threshold=0.5)  

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(OUTDIR / "train_history/train_history.csv", index=False)
torch.save(model_single.state_dict(), MODELSDIR / "MLP_model.pt")

ep=001 tr_loss=0.2076 | va_loss=0.1778 | tr R2=0.655 | va R2=0.612 | tr RMSE=0.588 | va RMSE=0.609 | tr MAE=0.459 | va MAE=0.481 | lr=1.00e-03 | bad_epochs=0
ep=010 tr_loss=0.0821 | va_loss=0.1790 | tr R2=0.853 | va R2=0.608 | tr RMSE=0.384 | va RMSE=0.613 | tr MAE=0.297 | va MAE=0.472 | lr=1.00e-03 | bad_epochs=3
Early stopping at epoch 13. Best epoch was 7 with val_loss=0.166744.
best epoch: 7
best val loss: 0.166744


In [11]:
m_te = eval_single_metrics_test(model_single, X_test_s, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

# m_te = eval_single_metrics_test(model_single, "regression", X_test_s, y_test,
#                                 y_mean, y_std, DEVICE)

# single_test_rows = []
# single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [12]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)
single_metrics_path = OUTDIR / "test_metrics/test_metrics.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

# single_metrics_df = pd.DataFrame(
#     single_test_rows,
#     columns=["output", "R2", "RMSE", "MAE"]
# )
# single_metrics_path = OUTDIR / "test_metrics_MLP_age.csv"
# single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [13]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.667628,7.862713,6.162862
